# Lecture 01: Computational Research Workflows

**PHYS690: Computational Methods for Physics Research**  
**Thursday, August 27, 2026**

Welcome. Today is a tour of the kind of work we will do this semester: turning a physics question into code, data, figures, checks, and a result that someone else can reproduce.

## How to use this workbook

This notebook is designed to run in [Google Colaboratory](https://colab.research.google.com/). It does not need any external files. All data are generated inside the notebook.

To run a cell, click in the cell and press `Shift` + `Enter`. Try to run the notebook from top to bottom, because later cells often use variables created earlier.

## Big picture

A computational research workflow is more than a calculation. A good workflow keeps track of:

- the scientific question;
- the assumptions or model;
- the input data or simulation settings;
- the code that transforms inputs into outputs;
- the figures, tables, and diagnostics used to interpret the result;
- enough documentation that the work can be rerun later.

This course is about building those habits while also learning practical numerical tools.

## Today's goals

By the end of today's lecture, you should have seen:

- Python as a tool for reproducible calculation;
- a small synthetic data set stored in a table;
- a publication-style plot with uncertainty bars;
- a simple model fit and residual check;
- a preview of numerical simulation;
- a preview of Monte Carlo methods;
- why the command line, Git, and project organization matter for research.

## Does the scientific Python stack work?

Run the next cell. The first line tells the notebook to show plots inline. The imports bring in the libraries we will use constantly: `numpy`, `pandas`, `matplotlib`, and pieces of `scipy`.

In [ ]:
%matplotlib inline

import platform
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

plt.style.use("seaborn-v0_8-whitegrid")
rng = np.random.default_rng(20260827)

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print("Scientific Python stack ready.")
print("pandas available:" , pd is not None)

## A tiny command-line preview

In Colab and Jupyter notebooks, a line beginning with `!` is sent to the system shell. Next lecture we will use a real terminal, but this gives us a small preview.

In [ ]:
!pwd
!python --version

# Example 1: A small measurement data set

Suppose we measure a damped oscillation. This could stand in for many real measurements: a mechanical oscillator, a detector response, a signal with decay, or any time-dependent quantity with noise.

The data below are synthetic, but the workflow is realistic: define a model, generate or load data, put the data in a table, and inspect the first few rows.

In [ ]:
def damped_cosine(t, amplitude, damping, angular_frequency, phase, offset):
    return amplitude * np.exp(-damping * t) * np.cos(angular_frequency * t + phase) + offset


true_parameters = {
    "amplitude": 1.25,
    "damping": 0.18,
    "angular_frequency": 2.70,
    "phase": 0.35,
    "offset": 0.05,
}

time_s = np.linspace(0.0, 8.0, 65)
sigma_m = 0.035 + 0.015 * time_s / time_s.max()
position_m = damped_cosine(time_s, **true_parameters) + rng.normal(0.0, sigma_m)

data = {
    "time_s": time_s,
    "position_m": position_m,
    "sigma_m": sigma_m,
}

if pd is not None:
    data_table = pd.DataFrame(data)
    display(data_table.head())
else:
    print("columns: time_s, position_m, sigma_m")
    print(np.column_stack([data["time_s"], data["position_m"], data["sigma_m"]])[:5])

A table is not just a convenience. Naming columns clearly is part of documenting the calculation. A future reader should not have to guess whether a column is time in seconds, time in nanoseconds, or a sample index.

In [ ]:
if pd is not None:
    display(data_table.describe())
else:
    for name, values in data.items():
        print(f"{name:>10}: min={values.min(): .4f}, mean={values.mean(): .4f}, max={values.max(): .4f}")

## Plot the data

A first plot is often the fastest way to find mistakes. Plot before you fit. Plot before you believe a number.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.errorbar(
    data["time_s"],
    data["position_m"],
    yerr=data["sigma_m"],
    fmt="o",
    ms=4,
    capsize=2,
    label="synthetic measurements",
)

ax.set_title("Damped oscillation data")
ax.set_xlabel("time (s)")
ax.set_ylabel("position (m)")
ax.legend()
plt.show()

### Try this

- Change the random seed in the setup cell and rerun the notebook.
- Increase the measurement uncertainties by changing `sigma_m`.
- Add or remove points by changing the number passed to `np.linspace`.

Each change should make you ask: did the result change for a physical reason, a numerical reason, or just because I changed the simulation?

# Example 2: Fit a model and inspect residuals

In Unit 2 we will spend much more time fitting models to data. Today we only want the outline: choose a model, estimate parameters, and check what the model missed.

In [ ]:
initial_guess = [1.0, 0.10, 2.4, 0.0, 0.0]
bounds = ([0.0, 0.0, 0.5, -np.pi, -1.0], [3.0, 1.0, 6.0, np.pi, 1.0])

best_fit, covariance = curve_fit(
    damped_cosine,
    data["time_s"],
    data["position_m"],
    p0=initial_guess,
    sigma=data["sigma_m"],
    absolute_sigma=True,
    bounds=bounds,
    maxfev=10000,
)

parameter_names = ["amplitude A", "damping gamma", "angular frequency omega", "phase phi", "offset c"]
uncertainties = np.sqrt(np.diag(covariance))

if pd is not None:
    fit_table = pd.DataFrame(
        {
            "parameter": parameter_names,
            "estimate": best_fit,
            "standard_uncertainty": uncertainties,
        }
    )
    display(fit_table)
else:
    for name, estimate, uncertainty in zip(parameter_names, best_fit, uncertainties):
        print(f"{name:>25}: {estimate: .5f} +/- {uncertainty:.5f}")

A fitted parameter without an uncertainty and a diagnostic plot is usually not enough. The next cell compares the fitted curve to the data and plots normalized residuals.

In [ ]:
fit_time = np.linspace(data["time_s"].min(), data["time_s"].max(), 400)
fit_curve = damped_cosine(fit_time, *best_fit)
model_at_data = damped_cosine(data["time_s"], *best_fit)
normalized_residuals = (data["position_m"] - model_at_data) / data["sigma_m"]

degrees_of_freedom = len(data["time_s"]) - len(best_fit)
reduced_chi2 = np.sum(normalized_residuals**2) / degrees_of_freedom

fig, (ax_data, ax_resid) = plt.subplots(
    2,
    1,
    figsize=(8, 6),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)

ax_data.errorbar(
    data["time_s"],
    data["position_m"],
    yerr=data["sigma_m"],
    fmt="o",
    ms=4,
    capsize=2,
    label="data",
)
ax_data.plot(fit_time, fit_curve, lw=2, label="best fit")
ax_data.set_ylabel("position (m)")
ax_data.legend()

ax_resid.axhline(0.0, color="black", lw=1)
ax_resid.errorbar(data["time_s"], normalized_residuals, yerr=1.0, fmt="o", ms=4, capsize=2)
ax_resid.set_xlabel("time (s)")
ax_resid.set_ylabel("resid. / sigma")

fig.suptitle(f"Model fit with reduced chi^2 = {reduced_chi2:.2f}")
fig.tight_layout()
plt.show()

### What to notice

- The fit returns numbers, but the plot tells us whether those numbers describe the data.
- Residuals should not show an obvious pattern if the model and uncertainty estimates are reasonable.
- A complete analysis should say what was fit, what assumptions were made, and what checks were performed.

# Example 3: Propagate uncertainty with random samples

Uncertainty is part of the result, not decoration added at the end. One common computational move is to draw possible parameter values and see how much the predicted curve changes.

In [ ]:
sample_rng = np.random.default_rng(690)
parameter_samples = sample_rng.multivariate_normal(best_fit, covariance, size=600, check_valid="ignore")
prediction_grid = np.linspace(0.0, 10.0, 500)

predictions = np.array([damped_cosine(prediction_grid, *sample) for sample in parameter_samples])
lower, middle, upper = np.percentile(predictions, [16, 50, 84], axis=0)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(data["time_s"], data["position_m"], yerr=data["sigma_m"], fmt="o", ms=4, capsize=2, label="data")
ax.plot(prediction_grid, middle, color="black", lw=2, label="median prediction")
ax.fill_between(prediction_grid, lower, upper, color="tab:green", alpha=0.25, label="central 68% band")
ax.set_title("Prediction band from sampled fit parameters")
ax.set_xlabel("time (s)")
ax.set_ylabel("position (m)")
ax.legend()
plt.show()

# Example 4: A numerical simulation

In Unit 3 we will solve differential equations and study numerical error. Here is a first look at using `scipy` to integrate a damped oscillator equation:

$$\frac{d^2x}{dt^2} + 2\gamma \frac{dx}{dt} + \omega_0^2 x = 0.$$

In [ ]:
def oscillator_rhs(t, state, damping, natural_frequency):
    x, v = state
    return [v, -2.0 * damping * v - natural_frequency**2 * x]


damping_demo = 0.12
natural_frequency_demo = 2.4
initial_state = [1.0, 0.0]
t_eval = np.linspace(0.0, 12.0, 600)

solution = solve_ivp(
    oscillator_rhs,
    t_span=(t_eval.min(), t_eval.max()),
    y0=initial_state,
    t_eval=t_eval,
    args=(damping_demo, natural_frequency_demo),
    rtol=1e-8,
    atol=1e-10,
)

x_num, v_num = solution.y
energy_like = 0.5 * (v_num**2 + natural_frequency_demo**2 * x_num**2)

fig, (ax_x, ax_e) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax_x.plot(solution.t, x_num)
ax_x.set_ylabel("x(t)")
ax_x.set_title("Numerical solution of a damped oscillator")

ax_e.plot(solution.t, energy_like, color="tab:red")
ax_e.set_xlabel("time (s)")
ax_e.set_ylabel("energy-like quantity")

fig.tight_layout()
plt.show()

This is a small example, but it already raises research questions: How accurate is the integrator? How should we choose tolerances? What quantity should be conserved or decrease? How do we know the result is not just a numerical artifact?

# Example 5: Monte Carlo estimation

In Unit 4 we will use random sampling for simulation, uncertainty propagation, and inference. The classic first example is estimating $\pi$ by throwing random points into a square and counting how many land inside a circle.

In [ ]:
mc_rng = np.random.default_rng(427)

points = mc_rng.uniform(-1.0, 1.0, size=(4000, 2))
radius_squared = points[:, 0] ** 2 + points[:, 1] ** 2
inside = radius_squared <= 1.0
pi_estimate = 4.0 * inside.mean()

n_values = np.unique(np.logspace(2, 5, 25, dtype=int))
running_estimates = []

for n in n_values:
    trial_points = mc_rng.uniform(-1.0, 1.0, size=(n, 2))
    trial_inside = np.sum(trial_points[:, 0] ** 2 + trial_points[:, 1] ** 2 <= 1.0)
    running_estimates.append(4.0 * trial_inside / n)

fig, (ax_points, ax_convergence) = plt.subplots(1, 2, figsize=(11, 4.5))

ax_points.scatter(points[inside, 0], points[inside, 1], s=4, alpha=0.45, label="inside")
ax_points.scatter(points[~inside, 0], points[~inside, 1], s=4, alpha=0.45, label="outside")
circle = plt.Circle((0.0, 0.0), 1.0, fill=False, color="black", lw=1.5)
ax_points.add_patch(circle)
ax_points.set_aspect("equal")
ax_points.set_title(f"pi estimate = {pi_estimate:.4f}")
ax_points.set_xlabel("x")
ax_points.set_ylabel("y")
ax_points.legend(markerscale=3)

ax_convergence.semilogx(n_values, running_estimates, "o-", label="Monte Carlo estimate")
ax_convergence.axhline(np.pi, color="black", lw=1, label="np.pi")
ax_convergence.set_xlabel("number of samples")
ax_convergence.set_ylabel("estimate of pi")
ax_convergence.set_title("Convergence is noisy")
ax_convergence.legend()

fig.tight_layout()
plt.show()

print(f"Monte Carlo estimate with {len(points)} points: pi = {pi_estimate:.5f}")
print(f"Reference value from NumPy:              pi = {np.pi:.5f}")

### Try this

- Increase `4000` to `40000` in the Monte Carlo cell.
- Change the random seed and rerun.
- Look at how the estimate changes with sample size.

Randomness can be useful, but only if we measure and communicate the uncertainty that comes with it.

# Why workflow matters

The code above was intentionally small, but a research version of this work quickly grows into multiple files and decisions:

- Which data are raw, and which are processed?
- Which script created each figure?
- Which parameter values were used?
- Which environment and package versions were used?
- Which results are final, and which were exploratory?
- How can a collaborator rerun the calculation?

This is why Unit 1 starts with command-line work, Git, Python environments, project organization, and documentation.

In [ ]:
workflow_note = {
    "question": "How well does a damped oscillation model describe noisy measurements?",
    "data": "Synthetic measurements generated inside this notebook with a fixed random seed.",
    "model": "Damped cosine with amplitude, damping, frequency, phase, and offset.",
    "checks": "Plot data, inspect residuals, and estimate uncertainty in predictions.",
    "next_step": "Move from a single notebook toward organized project folders, scripts, Git, and README files.",
}

for key, value in workflow_note.items():
    print(f"{key:>9}: {value}")

# For next lecture

Before the next lecture, install VS Code so we can use a native terminal, command-line tools, Git, and project folders on your own machine.

- Course resource: [Visual Studio Code (VS Code)](../resources/vscode.md)
- GitHub version: [Visual Studio Code (VS Code)](https://github.com/jrstevenjlab/gradcompphys/blob/main/resources/vscode.md)

Next time we will move from notebook cells toward a more complete research project structure.